# 🏥 Insurance RAG — Cross-Document Query Comparison (with Reranking)
**HDFC Ergo Optima Secure  vs  Care Insurance Supreme**

Both PDFs are extracted, chunked, and loaded into separate ChromaDB collections.
Every query runs against both simultaneously. We use a **two-stage retrieval pipeline**:
1. **Dense Retrieval (Bi-Encoder):** Quickly fetches the top $K$ candidate chunks.
2. **Reranking (Cross-Encoder):** Scores the exact relationship between the query and each candidate to surface the most contextually relevant chunks to the very top.

---

## 0 · Setup

In [ ]:
!pip install pymupdf4llm chromadb sentence-transformers rank_bm25 --quiet
!pip install langchain-core langchain-text-splitters langchain-chroma langchain-huggingface langchain-community langchain-google-genai --quiet 
print('✅ Dependencies installed')

In [ ]:
import sys, os, json, textwrap, time
from pathlib import Path

# ── If cloned from GitHub ─────────────────────────────────────────────────
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

# ── If cloned from GitHub ─────────────────────────────────────────────────
!git clone https://{token}@github.com/falcon978/Insurance-RAG
%cd Insurance-RAG

from rag_ingestion.pipeline import ExtractionPipeline
print('✅ Package imported')

In [ ]:
import importlib

# Import the modules
from rag_ingestion import pipeline, extractor, chunker, cleaner, models, patterns

# Reload the modules to reflect any code changes
importlib.reload(patterns)
importlib.reload(models)
importlib.reload(pipeline)
importlib.reload(extractor)
importlib.reload(chunker)
importlib.reload(cleaner)
# Import the specific classes/functions after reload
from rag_ingestion.pipeline import ExtractionPipeline
from rag_ingestion.extractor import PDFExtractor
from rag_ingestion.chunker import MarkdownHierarchicalChunker
from rag_ingestion.cleaner import clean_block_text

print("Modules reloaded successfully!")

---
## 1 · Download Both PDFs

In [ ]:
import urllib.request

PDFS = {
    "optima_secure": {
        "name"  : "HDFC Ergo Optima Secure",
        "url"   : (
            "https://customer-portal-assets.hdfcergo.com/assets/v2/docs/"
            "default-source/downloads/policy-wordings/health/"
            "optima-secure-revision/optima-secure-revision-pw-647504209314.pdf"
        ),
        "path"  : "hdfc_optima_secure.pdf",
    },
    "care_supreme": {
        "name"  : "Care Insurance Supreme",
        "url"   : (
            "https://cms.careinsurance.com/cms/public/uploads/download_center/"
            "care-supreme---policy-terms-&-conditions-(effective-from-19-march-2025).pdf"
            "?rv=0.86869200%201775054695"
        ),
        "path"  : "care_supreme.pdf",
    },
}

for key, info in PDFS.items():
    if not Path(info['path']).exists():
        print(f"Downloading {info['name']} …")
        try:
            req = urllib.request.Request(
                info['url'],
                headers={'User-Agent': 'Mozilla/5.0'}
            )
            with urllib.request.urlopen(req, timeout=60) as r:
                Path(info['path']).write_bytes(r.read())
            size = Path(info['path']).stat().st_size // 1024
            print(f"  ✅ {info['path']}  ({size} KB)")
        except Exception as e:
            print(f"  ❌ Failed: {e}")
            print(f"  → Upload {info['path']} manually using the cell below")
    else:
        print(f"✅ {info['path']} already exists")

In [ ]:
# ── Manual upload fallback (run only if download failed) ──────────────────
# from google.colab import files
# uploaded = files.upload()
# Rename the uploaded files to match PDFS paths above if needed

---
## 2 · Extract & Chunk Both PDFs

In [ ]:
CHUNK_SIZE = 1200
OVERLAP    = 150

results = {}

for key, info in PDFS.items():
    if not Path(info['path']).exists():
        print(f"⚠️  {info['path']} not found — skipping")
        continue

    print(f"\n{'='*60}")
    print(f"Extracting & Indexing: {info['name']}")
    print('='*60)

    # UPDATED: Pass a unique collection_name to the pipeline
    result = ExtractionPipeline(
        pdf_path        = info['path'],
        chunk_size      = CHUNK_SIZE,
        chunk_overlap   = OVERLAP,
        collection_name = f"insurance_{key}", # <-- Separates the databases!
        device          = "cpu"
    ).run()

    results[key] = result
    print(f"Stats: {result.stats}")

---
## 3 · Load Models & Index into ChromaDB

In [ ]:
# ── Cell 3: Load Embeddings and Vector Stores ─────────────────────────────

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

print("Loading Bi-Encoder (Embeddings)...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("Connecting to ChromaDB via LangChain...")
hdfc_vector_store = Chroma(
    collection_name="insurance_optima_secure",
    embedding_function=embeddings,
    persist_directory="./chroma_data"
)

care_vector_store = Chroma(
    collection_name="insurance_care_supreme",
    embedding_function=embeddings,
    persist_directory="./chroma_data"
)

print(f"✅ HDFC Store: {hdfc_vector_store._collection.count()} chunks")
print(f"✅ Care Store: {care_vector_store._collection.count()} chunks")

---
## 4 · Query Interface with Two-Stage Retrieval

In [ ]:
# ── Cell 4: Import Retrieval Architecture ─────────────────────────────

import logging
from rag.retriever import DocumentSearch
from rag.rerankers import ContextReranker

# Keep notebook output clean
logger = logging.getLogger()
logger.setLevel(logging.ERROR) 

print("Initializing Reranker (Cross-Encoder)...")
# We load this once because the model is heavy
reranker = ContextReranker(device="cpu") 

print("✅ Architecture imported from backend.")

In [ ]:
# ── Cell 5: Notebook Query Wrappers ───────────────────────────────────

def retrieve(query: str, retrieve_top_k: int = 15, rerank_top_k: int = 3, apply_rerank: bool = True) -> dict:
    """Fetches documents dynamically using the backend RAG components."""
    output = {}
    
    # Dynamically initialize engines to respect the custom retrieve_top_k
    hdfc_search = DocumentSearch(hdfc_vector_store, strategy="hybrid", top_k=retrieve_top_k)
    care_search = DocumentSearch(care_vector_store, strategy="hybrid", top_k=retrieve_top_k)
    
    for key, search_engine in [('optima_secure', hdfc_search), ('care_supreme', care_search)]:
        broad_docs = search_engine.search(query)
        if apply_rerank and broad_docs:
            # We pass return_scores=True purely so the notebook can print the math!
            output[key] = reranker.rerank(query, broad_docs, top_k=rerank_top_k, return_scores=True)
        else:
            output[key] = [(doc, None) for doc in broad_docs[:rerank_top_k]]
    return output


def show(query: str, retrieve_top_k: int = 15, rerank_top_k: int = 3, apply_rerank: bool = True, text_preview: int = 400):
    """Pretty-prints the retrieved chunks for easy inspection."""
    print(f"\n{'▓'*70}\n  QUERY: {query}\n{'▓'*70}")
    
    raw = retrieve(query, retrieve_top_k=retrieve_top_k, rerank_top_k=rerank_top_k, apply_rerank=apply_rerank)

    for key, hits in raw.items():
        print(f"\n{'─'*70}\n  📄 {PDFS[key]['name']}\n{'─'*70}")
        if not hits: continue

        for rank, (doc, rerank_score) in enumerate(hits, 1):
            r_score_str = f"{rerank_score:.2f}" if rerank_score is not None else "N/A"
            section = doc.metadata.get('section', '')[:40]
            pages   = f"p.{doc.metadata.get('page_start')}–{doc.metadata.get('page_end')}"
            
            # Format preview text cleanly
            preview = doc.page_content[:text_preview].replace('\n', ' ') 
            if len(doc.page_content) > text_preview:
                preview += ' …'

            print(f"\n  [{rank}] Rerank Score: {r_score_str}  |  {pages}")
            print(f"       Section : {section}")
            print(f"       Heading : {doc.metadata.get('heading', '')[:60]}")
            print(f"       Text    : {preview}")

print("\n✅ Query wrappers ready! Use show('your query') to test.")

---
## 5 · Run Your Queries
Edit the query string in any cell below and run it.  
Add as many cells as you need — `show()` is all you need.

In [ ]:
# ── Your query here ───────────────────────────────────────────────────────
show(
    "Does this policy cover robotic surgery?", 
    rerank_top_k=3, 
    retrieve_top_k=25
)

In [ ]:
# ── With section filter ───────────────────────────────────────────────────
show("What is the waiting period for pre-existing diseases?", section_filter="waiting")

In [ ]:
# ── Compare Vector vs Reranker Output ─────────────────────────────────────
q = "Are maternity expenses covered?"

print("\n--- WITHOUT RERANKING (Raw Hybrid Results) ---")
show(q, apply_rerank=False, rerank_top_k=1)

print("\n--- WITH RERANKING (Cross-Encoder Optimized) ---")
show(q, apply_rerank=True, rerank_top_k=1)

In [ ]:
# ── Golden Dataset Inspector ──────────────────────────────────────────────

def inspect_for_dataset(query: str, policy_key: str, rerank_top_k: int = 4):
    """
    Prints raw text and metadata of top reranked chunks.
    Use this to write your ground_truth answers.
    """
    print(f"🛠️  INSPECTOR: Fetching top {rerank_top_k} chunks for '{query}'")
    print(f"📁  Policy: {PDFS[policy_key]['name']}\n")
    
    # retrieve() returns {policy_key: [(doc, score), ...]}
    raw_results = retrieve(query, rerank_top_k=rerank_top_k, apply_rerank=True)
    hits = raw_results.get(policy_key, [])
    
    if not hits:
        print("No chunks found.")
        return
        
    for rank, (doc, rerank_score) in enumerate(hits, 1):
        r_score = f"{rerank_score:.3f}" if rerank_score is not None else "N/A"
        
        print(f"CHUNK RANK [{rank}] {'='*60}")
        print(f"🔍 Rerank Score: {r_score}")
        print(f"🔖 Section     : {doc.metadata.get('section', 'None')}")
        print(f"🏷️  Heading     : {doc.metadata.get('heading', 'None')}")
        print(f"📄 Pages       : {doc.metadata.get('page_start')} - {doc.metadata.get('page_end')}")
        print("-" * 75)
        print(doc.page_content.strip())
        print("=" * 75)
        print("\n")

# Run it
inspect_for_dataset("What is the waiting period for maternity benefits?", policy_key="optima_secure")

---
## 6 · Retrieval Quality Diagnostics
Run these after your queries to understand what's working and what isn't.

In [ ]:
!pip install numpy scikit-learn --quiet
print("✅ Diagnostic dependencies installed")

In [ ]:
# ── Score distribution for a query ───────────────────────────────────────
def score_distribution(query: str, retrieve_top_k: int = 15, rerank_top_k: int = 10):
    # Pass BOTH parameters to the retrieve function
    raw = retrieve(
        query, 
        retrieve_top_k=retrieve_top_k, 
        rerank_top_k=rerank_top_k, 
        apply_rerank=True
    )
    
    print(f'Query: "{query}"')
    print(f'Config: [Retrieve Pool: {retrieve_top_k} | Rerank Show: {rerank_top_k}]\n')
    
    for key, hits in raw.items():
        print(f"{PDFS[key]['name']}:")
        if not hits:
            print("  No results found.")
            continue
            
        for rank, (doc, rerank_score) in enumerate(hits, 1):
            # Normalizing logit for visual bar mapping (-10 to +10 range)
            norm_score = max(0, min(1, (rerank_score + 10) / 20)) if rerank_score is not None else 0
            bar = '█' * int(norm_score * 20)
            
            # Use page and heading for better diagnostic context
            page = doc.metadata.get('page_start', '?')
            heading = doc.metadata.get('heading', 'No Heading')[:40]
            r_str = f"{rerank_score:+.2f}" if rerank_score is not None else "N/A"
            
            print(f"  [{rank}] R:{r_str:<6} | {bar:<20} [p.{page}] {heading}")
        print()

# Now you can tune the 'net' size during diagnostics
score_distribution("What are the room rent limits?", retrieve_top_k=30, rerank_top_k=5)

In [ ]:
# ── Check which sections are being retrieved for a query ──────────────────
def section_coverage(query: str, retrieve_top_k: int = 20, rerank_top_k: int = 10):
    raw = retrieve(query, retrieve_top_k=retrieve_top_k, rerank_top_k=rerank_top_k, apply_rerank=True)
    
    print(f'Query: "{query}"\n')
    for key, hits in raw.items():
        print(f"{PDFS[key]['name']}:")
        if not hits: continue
        
        for rank, (doc, rerank_score) in enumerate(hits, 1):
            r_str = f"{rerank_score:+.2f}" if rerank_score is not None else "N/A"
            section = str(doc.metadata.get('section', 'None'))[:35]
            heading = str(doc.metadata.get('heading', 'None'))[:35]
            print(f"  {r_str:>6}  [{section:<35}] > {heading}")
        print()

section_coverage("How is day care treatment handled?", retrieve_top_k=25)

In [ ]:
def check_rerank_impact(query: str, policy_key: str, retrieve_top_k: int = 20):
    """
    Measures the 'Value-Add' of the Cross-Encoder by identifying the original 
    rank of the final winner.
    
    Why this matters: 
    - If the winner was originally Rank 1, your Embeddings/BM25 are already strong.
    - If the winner was Rank 15, the Reranker 'saved' the query by surfacing a 
      chunk the Vector search almost missed.
    """
    # 1. Get raw Hybrid results (Simulating no rerank)
    # We set rerank_top_k to the full retrieve_top_k to see the whole list
    raw_hybrid = retrieve(query, retrieve_top_k=retrieve_top_k, rerank_top_k=retrieve_top_k, apply_rerank=False)[policy_key]
    
    # 2. Get the actual Reranked result (The winner)
    reranked = retrieve(query, retrieve_top_k=retrieve_top_k, rerank_top_k=1, apply_rerank=True)[policy_key]
    
    if not reranked: 
        print("No results found.")
        return
    
    winner_text = reranked[0][0].page_content
    
    # 3. Find where the winner was sitting in the initial hybrid results
    original_rank = "Not in initial pool"
    for idx, (doc, _) in enumerate(raw_hybrid):
        if doc.page_content == winner_text:
            original_rank = idx + 1
            break
            
    print(f"📊 Rerank Impact Analysis: '{query}'")
    print(f"🏆 The final top chunk was originally ranked #{original_rank} by Hybrid Search.")
    
    if isinstance(original_rank, int) and original_rank > 5:
        print(f"💡 High Impact: The Reranker improved this result by {original_rank - 1} positions!")
    elif original_rank == 1:
        print("💡 Low Impact: Hybrid search and Reranker agreed on the best chunk.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def check_context_redundancy(query: str, policy_key: str, rerank_top_k: int = 4):
    """
    Calculates semantic overlap between the top K chunks sent to the LLM.
    
    Why this matters:
    - Insurance docs often repeat headers or table structures.
    - If top chunks are >90% similar, you are sending redundant data to the LLM.
    - Diversity in context ensures the LLM sees different angles of the policy.
    """
    # Get the final chunks that would be sent to the LLM
    results = retrieve(query, rerank_top_k=rerank_top_k, apply_rerank=True)
    docs = [hit[0] for hit in results.get(policy_key, [])]
    
    if len(docs) < 2:
        print("Not enough documents to calculate redundancy.")
        return

    # Embed the actual text content to check for mathematical similarity
    texts = [doc.page_content for doc in docs]
    vecs = embeddings.embed_documents(texts)
    
    # Compute Cosine Similarity matrix (1.0 = identical, 0.0 = completely different)
    sim_matrix = cosine_similarity(vecs)
    
    # We only care about the values above the diagonal (avoiding self-comparison)
    upper_tri = sim_matrix[np.triu_indices(len(sim_matrix), k=1)]
    avg_sim = np.mean(upper_tri)
    
    print(f"📊 Context Redundancy: {PDFS[policy_key]['name']}")
    print(f"  Average Inter-chunk Similarity: {avg_sim:.3f}")
    
    if avg_sim > 0.85:
        print("  ⚠️ Warning: High redundancy detected! The LLM is receiving very similar chunks.")
        print("  Try: Reducing chunk overlap or increasing retrieve_top_k for more variety.")
    else:
        print("  ✅ Good diversity: Each chunk likely provides unique information to the LLM.")

check_context_redundancy("room rent limits", "optima_secure")

In [ ]:
# ── Chunk size stats per insurer ──────────────────────────────────────────
# Quick sanity check that chunks are well-sized

for key, result in results.items():
    chunks = result.chunks
    char_counts = [len(c.text) for c in chunks]
    print(f"📊 {PDFS[key]['name']} Statistics:")
    print(f"  Total chunks : {len(chunks)}")
    print(f"  Mean chars   : {sum(char_counts)//len(char_counts)}")
    print(f"  Mean tokens~ : {sum(char_counts)//(len(char_counts)*4)}")
    print("-" * 30)

---
## 7 · LLM Generation (The Legal Assistant)
This stage takes the highly specific chunks retrieved by the Cross-Encoder and feeds them into an LLM. 
The LLM is strictly constrained by a system prompt to prevent hallucinations and to explicitly advise the user on what to ask the insurer if a clause is "silent" (missing).

In [ ]:
# Install the necessary LangChain and LLM packages
!pip install langchain-google-genai --quiet
print('✅ LLM dependencies installed')

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import display, Markdown

# 1. Setup your API Key
from google.colab import userdata
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = input("Enter your Google Gemini API Key: ")

# Initialize the LLM
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0, api_key=GEMINI_API_KEY)

# 2. Define the Strict System Guardrails
system_prompt = """
You are an expert Insurance Policy Analyst AI. Compare how these two policies handle the user's query based ONLY on the provided contexts.

Policy A Context:
{hdfc_context}

Policy B Context:
{care_context}

YOUR STRICT RULES:
1. DIRECT COMPARISON & CITATIONS: Compare the policies directly. You MUST cite the exact policy name and page number for every fact stated (e.g., '[Source: Care Supreme, Page 14]').
2. ABSENCE OF TERM: If one or both documents DO NOT explicitly mention the queried term, DO NOT guess or hallucinate. State clearly which policy lacks the explicit mention. Do not invent citations for missing information.
3. CONTEXTUAL REASONING: If exact terms are missing, analyze provided 'catch-all' clauses or general exclusions to infer potential handling.
4. THE 'FINE PRINT' BATTLE: Extract and contrast exact numerical limits, sub-limits, co-payments, and waiting periods. Clearly state if one policy is mathematically more favorable based on the provided text.
5. CONDITIONAL GAP-FILLING QUESTIONS: ONLY IF information is missing, ambiguous, or completely absent from one or both policies, provide 1-2 targeted questions the user should ask the respective insurer to fill the gap. Do NOT provide questions if the provided contexts fully answer the query.
6. FORMATTING: Use Markdown. Create a clear header for each insurer and conclude with a 'Comparison Summary'.
"""

# Create the LangChain Prompt Template
prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "USER QUERY: {query}")
])

# Create the Generation Chain
analysis_chain = prompt_template | llm

print("✅ LLM Pipeline and Guardrails Ready")

In [ ]:
def generate_policy_comparison(query: str, retrieve_top_k: int = 15, rerank_top_k: int = 3):
    print(f"🔍 Retrieving context for: '{query}'...")

    # PASSING THE VARIABLES DOWN TO RETRIEVE()
    raw_results = retrieve(query, retrieve_top_k=retrieve_top_k, rerank_top_k=rerank_top_k, apply_rerank=True)

    def format_context(hits, policy_name):
        if not hits:
            return "No relevant clauses found in the document."

        context_str = ""
        for rank, (doc, rerank_score) in enumerate(hits, 1):
            page = doc.metadata.get('page_start', 'Unknown')
            context_str += f"\n\n--- [Source: {policy_name}, Page {page}] ---\n"
            context_str += f"{doc.page_content}\n"
        return context_str

    hdfc_text = format_context(raw_results.get('optima_secure', []), "HDFC Optima Secure")
    care_text = format_context(raw_results.get('care_supreme', []), "Care Supreme")

    print("🧠 Analyzing legal text with LLM...\n")
    print("="*80)

    response = analysis_chain.invoke({
        "query": query,
        "hdfc_context": hdfc_text,
        "care_context": care_text
    })

    if isinstance(response.content, list) and all(isinstance(item, dict) and 'text' in item for item in response.content):
        markdown_text = "".join([item['text'] for item in response.content])
        display(Markdown(markdown_text))
    else:
        content = getattr(response, 'content', response) 
        display(Markdown(str(content)))

print("✅ Generation function ready.")

In [ ]:
# Test the end-to-end pipeline!
query = "Are there any limits, caps, or proportionate deductions applied to room rent or ICU charges?"
generate_policy_comparison(query)

In [ ]:
# ── Single Policy Q&A Generation ──────────────────────────────────────────

# 1. Define the Single Policy Prompt
single_system_prompt = """
You are an expert Insurance Policy Analyst AI. Answer the user's query using ONLY the provided policy context.

YOUR STRICT RULES:
1. EXPLICIT MATCH & CITATIONS: If the document explicitly answers the query, explain the coverage. You MUST cite the exact page number provided in the context blocks for every fact (e.g., 'Room rent is capped at 1% [Source: HDFC Optima Secure, Page 12]').
2. ABSENCE OF TERM: If the document DOES NOT explicitly mention the queried term, DO NOT guess, hallucinate, assume coverage, or invent citations. State clearly that it is not explicitly mentioned in the retrieved text.
3. CONTEXTUAL REASONING: If the exact term is missing, check the provided context for 'catch-all' clauses (e.g., 'Modern Treatments', 'General Exclusions', 'Definitions') and explain how they *might* apply.
4. HIDDEN CATCHES (WAITING PERIODS & LIMITS): Proactively check for and state any waiting periods, co-payments, or sub-limits attached to the queried benefit, even if the user didn't explicitly ask for them.
5. GAP-FILLING QUESTIONS: If there is ambiguity, absence of information, or complex conditions, explicitly advise the user to contact the insurer. Provide 2-3 specific questions the user should ask. CRITICAL: These questions MUST target the missing or ambiguous information. Do NOT suggest asking questions that are already clearly answered in the provided context.
"""

# ── Single Policy Q&A Generation ──────────────────────────────────────────

single_prompt_template = ChatPromptTemplate.from_messages([
    ("system", single_system_prompt), # Assuming single_system_prompt is still defined above
    ("human", "Context:\n{context}\n\nUSER QUERY: {query}")
])

single_analysis_chain = single_prompt_template | llm

def generate_single_policy(query: str, policy_key: str, retrieve_top_k: int = 15, rerank_top_k: int = 3):
    print(f"🔍 Retrieving context for: '{query}' in {PDFS[policy_key]['name']}...")
    
    # PASSING THE VARIABLES DOWN TO RETRIEVE()
    raw_results = retrieve(query, retrieve_top_k=retrieve_top_k, rerank_top_k=rerank_top_k, apply_rerank=True)
    hits = raw_results.get(policy_key, [])
    
    context_str = ""
    if not hits:
        context_str = "No relevant clauses found in the document."
    else:
        for rank, (doc, rerank_score) in enumerate(hits, 1):
            page = doc.metadata.get('page_start', 'Unknown')
            context_str += f"\n\n--- [Source: {PDFS[policy_key]['name']}, Page {page}] ---\n"
            context_str += f"{doc.page_content}\n"
            
    print("🧠 Analyzing legal text with LLM...\n")
    print("="*80)
    
    response = single_analysis_chain.invoke({
        "context": context_str,
        "query": query
    })
    
    if isinstance(response.content, list) and all(isinstance(item, dict) and 'text' in item for item in response.content):
        markdown_text = "".join([item['text'] for item in response.content])
        display(Markdown(markdown_text))
    else:
        content = getattr(response, 'content', response) 
        display(Markdown(str(content)))

# --- Test the Single Policy Query ---
query = "What is the waiting period for maternity benefits?"
# Now you can tune it right here!
generate_single_policy(query, policy_key="optima_secure", retrieve_top_k=20, rerank_top_k=4)